# Human Gene Essentiality V2 - FINAL (No Leakage)
## Strict train/test separation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded!")

In [ ]:
DATA_DIR = '/content/drive/MyDrive/depmap_data'
print("Files:")
for f in os.listdir(DATA_DIR):
    print(f"  {f}")

In [ ]:
# Load CRISPR
print("Loading CRISPR...")
crispr_raw = pd.read_csv(os.path.join(DATA_DIR, 'CRISPRGeneEffect.csv'), index_col=0)
print(f"Shape: {crispr_raw.shape} (cells x genes)")
print(f"Index type: {type(crispr_raw.index[0])}")
print(f"Sample indices: {list(crispr_raw.index[:3])}")

In [ ]:
# Load Expression
print("\nLoading Expression...")
expr_file = [f for f in os.listdir(DATA_DIR) if 'expression' in f.lower() and 'crispr' not in f.lower()][0]
print(f"File: {expr_file}")

expr_full = pd.read_csv(os.path.join(DATA_DIR, expr_file))
print(f"Raw shape: {expr_full.shape}")

In [ ]:
# Find ModelID column and gene start
model_col = None
for col in expr_full.columns:
    try:
        if str(expr_full[col].iloc[0]).startswith('ACH-'):
            model_col = col
            print(f"ModelID column: '{col}'")
            break
    except: pass

gene_start = 0
for i, col in enumerate(expr_full.columns):
    if ' (' in str(col) and ')' in str(col):
        gene_start = i
        print(f"Gene columns start at {i}")
        break

# Create expression df with ModelID as index
expr_clean = expr_full.iloc[:, gene_start:].copy()
expr_clean.index = expr_full[model_col].values
print(f"Expression cleaned: {expr_clean.shape}")
print(f"Index type: {type(expr_clean.index[0])}")
print(f"Sample indices: {list(expr_clean.index[:3])}")

In [ ]:
# DEBUG: Check for duplicates
print(f"\nCRISPR duplicate indices: {crispr_raw.index.duplicated().sum()}")
print(f"Expression duplicate indices: {expr_clean.index.duplicated().sum()}")

# Remove duplicates if any (keep first)
if expr_clean.index.duplicated().sum() > 0:
    print("Removing expression duplicates...")
    expr_clean = expr_clean[~expr_clean.index.duplicated(keep='first')]
    print(f"Expression after dedup: {expr_clean.shape}")

In [ ]:
# Find common cells
crispr_cells = set(crispr_raw.index)
expr_cells = set(expr_clean.index)

print(f"CRISPR cells: {len(crispr_cells)}")
print(f"Expression cells: {len(expr_cells)}")

common_cells = sorted(list(crispr_cells & expr_cells))
print(f"Common cells: {len(common_cells)}")

# Verify all common_cells exist in both
assert all(c in crispr_raw.index for c in common_cells), "Some common_cells not in CRISPR!"
assert all(c in expr_clean.index for c in common_cells), "Some common_cells not in Expression!"
print("✅ All common_cells verified in both datasets")

In [ ]:
# Find common genes
def get_symbol(name):
    if ' (' in str(name):
        return str(name).split(' (')[0].upper()
    return str(name).upper()

crispr_sym = {get_symbol(c): c for c in crispr_raw.columns}
expr_sym = {get_symbol(c): c for c in expr_clean.columns}

common_symbols = sorted(list(set(crispr_sym.keys()) & set(expr_sym.keys())))
print(f"CRISPR genes: {len(crispr_sym)}")
print(f"Expression genes: {len(expr_sym)}")
print(f"Common gene symbols: {len(common_symbols)}")

In [ ]:
# Get original column names for common symbols
crispr_gene_cols = [crispr_sym[g] for g in common_symbols]
expr_gene_cols = [expr_sym[g] for g in common_symbols]

print(f"CRISPR gene columns: {len(crispr_gene_cols)}")
print(f"Expression gene columns: {len(expr_gene_cols)}")

In [ ]:
# Create aligned subsets - EXPLICIT about using same cells
print("\nCreating aligned subsets...")

# Subset CRISPR: common_cells x crispr_gene_cols
crispr_sub = crispr_raw.loc[common_cells, crispr_gene_cols].copy()
print(f"CRISPR subset: {crispr_sub.shape}")

# Subset Expression: common_cells x expr_gene_cols  
expr_sub = expr_clean.loc[common_cells, expr_gene_cols].copy()
print(f"Expression subset: {expr_sub.shape}")

# Verify same shape
assert crispr_sub.shape == expr_sub.shape, f"Shape mismatch: {crispr_sub.shape} vs {expr_sub.shape}"

In [ ]:
# Rename columns to common symbols
crispr_sub.columns = common_symbols
expr_sub.columns = common_symbols

# Verify indices match
assert list(crispr_sub.index) == list(expr_sub.index), "Row indices don't match!"
assert list(crispr_sub.columns) == list(expr_sub.columns), "Column names don't match!"

print("✅ Indices aligned!")

In [ ]:
# Transpose: genes x cells
crispr_df = crispr_sub.T
expr_df = expr_sub.T

print(f"CRISPR final: {crispr_df.shape} (genes x cells)")
print(f"Expression final: {expr_df.shape} (genes x cells)")

assert crispr_df.shape == expr_df.shape
assert list(crispr_df.index) == list(expr_df.index)
assert list(crispr_df.columns) == list(expr_df.columns)
print("✅ Perfect alignment!")

In [ ]:
# Binarize
binary_mat = (crispr_df < -0.5).astype(int)
print(f"Binary: {binary_mat.shape}")
print(f"Essential rate: {binary_mat.sum().sum() / binary_mat.size:.1%}")

In [ ]:
# STRICT TRAIN/TEST SPLIT
all_cells = list(binary_mat.columns)
np.random.seed(42)
shuffled = all_cells.copy()
np.random.shuffle(shuffled)

N_TRAIN = 200
N_TEST = 100

TRAIN_CELLS = shuffled[:N_TRAIN]
TEST_CELLS = shuffled[N_TRAIN:N_TRAIN+N_TEST]

print(f"Train cells: {len(TRAIN_CELLS)}")
print(f"Test cells: {len(TEST_CELLS)}")
print(f"Overlap: {len(set(TRAIN_CELLS) & set(TEST_CELLS))}")

In [ ]:
# Gene classification using ONLY TRAIN cells
gene_consensus_train = binary_mat[TRAIN_CELLS].mean(axis=1)

print(f"Pan-essential (≥90%):  {(gene_consensus_train >= 0.9).sum()}")
print(f"Common (50-90%):       {((gene_consensus_train >= 0.5) & (gene_consensus_train < 0.9)).sum()}")
print(f"Context-dep (10-50%):  {((gene_consensus_train >= 0.1) & (gene_consensus_train < 0.5)).sum()}")
print(f"Rarely ess (0-10%):    {((gene_consensus_train > 0) & (gene_consensus_train < 0.1)).sum()}")
print(f"Non-essential (0%):    {(gene_consensus_train == 0).sum()}")

In [ ]:
# V2 Predictor - NO LEAKAGE
class HedgeFundV2_NoLeak:
    def __init__(self):
        self.ml = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1)
        self.scaler = StandardScaler()
        self.fitted = False
        self.train_cells = None
        
    def _feat(self, bmat, emat, gene, cell, ref_cells):
        # All stats from reference cells only
        ref = [c for c in ref_cells if c != cell]
        cons = bmat.loc[gene, ref].mean()
        var = bmat.loc[gene, ref].var()
        
        expr_ref = emat.loc[gene, ref]
        expr_m = expr_ref.mean()
        expr_s = expr_ref.std() + 1e-6
        
        expr = emat.loc[gene, cell]
        expr_z = (expr - expr_m) / expr_s
        expr_p = (expr_ref < expr).mean()
        
        return [cons, var, expr, expr_z, expr_p, cons*expr, cons**2, expr**2]
    
    def fit(self, bmat, emat, train_cells):
        print("Training V2 (no leakage)...")
        self.train_cells = train_cells
        
        cons = bmat[train_cells].mean(axis=1)
        ctx_genes = cons[(cons > 0.1) & (cons < 0.9)].index.tolist()
        print(f"  Context genes: {len(ctx_genes)}")
        
        X, y = [], []
        for c in tqdm(train_cells, desc="  Training"):
            for g in ctx_genes:
                try:
                    f = self._feat(bmat, emat, g, c, train_cells)
                    if not np.isnan(f).any():
                        X.append(f)
                        y.append(bmat.loc[g, c])
                except: pass
        
        X, y = np.array(X), np.array(y)
        print(f"  Samples: {len(X):,}, Pos rate: {y.mean():.1%}")
        
        self.scaler.fit(X)
        self.ml.fit(self.scaler.transform(X), y)
        
        yp = self.ml.predict(self.scaler.transform(X))
        print(f"  Train acc: {accuracy_score(y, yp):.3f}, bal: {balanced_accuracy_score(y, yp):.3f}")
        
        names = ['cons', 'var', 'expr', 'expr_z', 'expr_p', 'c*e', 'c^2', 'e^2']
        print("  Top features:", [(n, f"{i:.2f}") for n, i in sorted(zip(names, self.ml.feature_importances_), key=lambda x: -x[1])[:3]])
        
        self.fitted = True
        return self
    
    def predict(self, bmat, emat, cell):
        cons = bmat[self.train_cells].mean(axis=1)
        preds = pd.Series(0, index=bmat.index)
        preds[cons >= 0.9] = 1
        
        ctx_genes = preds[(cons > 0.1) & (cons < 0.9)].index.tolist()
        
        if ctx_genes and self.fitted:
            X, valid = [], []
            for g in ctx_genes:
                try:
                    f = self._feat(bmat, emat, g, cell, self.train_cells)
                    if not np.isnan(f).any():
                        X.append(f)
                        valid.append(g)
                except: pass
            if X:
                for g, p in zip(valid, self.ml.predict(self.scaler.transform(np.array(X)))):
                    preds[g] = p
        return preds

print("V2 ready!")

In [ ]:
# Train
pred = HedgeFundV2_NoLeak()
pred.fit(binary_mat, expr_df, TRAIN_CELLS)

In [ ]:
# Evaluate
yt, yp = [], []
for c in tqdm(TEST_CELLS, desc="Testing"):
    yt.extend(binary_mat[c].values)
    yp.extend(pred.predict(binary_mat, expr_df, c).values)

yt, yp = np.array(yt), np.array(yp)

print(f"\n{'='*60}")
print(f"V2 RESULTS (NO LEAKAGE)")
print(f"{'='*60}")
print(f"Accuracy:   {accuracy_score(yt, yp):.3f}")
print(f"Balanced:   {balanced_accuracy_score(yt, yp):.3f}")
print(f"F1:         {f1_score(yt, yp):.3f}")

In [ ]:
# By category
v1 = {'Pan-essential (≥90%)': 0.982, 'Common (50-90%)': 0.434, 
      'Context-dep (10-50%)': 0.439, 'Rarely ess (0-10%)': 0.991, 'Non-essential (0%)': 1.0}

gc = gene_consensus_train
cats = {
    'Pan-essential (≥90%)': gc >= 0.9,
    'Common (50-90%)': (gc >= 0.5) & (gc < 0.9),
    'Context-dep (10-50%)': (gc >= 0.1) & (gc < 0.5),
    'Rarely ess (0-10%)': (gc > 0) & (gc < 0.1),
    'Non-essential (0%)': gc == 0
}

print("\nV1 → V2 (No Leakage):")
print("-"*70)
v2_results = {}
for name, mask in cats.items():
    n = mask.sum()
    if n > 0:
        m = np.tile(mask.values, len(TEST_CELLS))
        v2_acc = accuracy_score(yt[m], yp[m])
        d = v2_acc - v1[name]
        e = "✅" if d > 0.05 else "➡️" if d > -0.05 else "❌"
        v2_results[name] = v2_acc
        print(f"{name:25s}: {v1[name]:.1%} → {v2_acc:.1%} ({d:+.1%}) {e}")

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(cats))
w = 0.35

v1_vals = [v1[c] for c in cats]
v2_vals = [v2_results.get(c, 0) for c in cats]

ax.bar(x - w/2, v1_vals, w, label='V1', color='#ff6b6b')
ax.bar(x + w/2, v2_vals, w, label='V2 (No Leak)', color='#4ecdc4')
ax.axhline(0.5, color='gray', ls='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(cats.keys(), rotation=45, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('V1 vs V2')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()